In [1]:
from typing import Any

import torch
import torch.nn as nn
import numpy as np

In [2]:
# class DGMLayer_(nn.Module):
#     """
#     Georgias Detorakis (2024): Practical Aspects on Solving Differential Equations Using Deep Learning: A Primer

#     """

#     def __init__(self, input_dim=1, hidden_size=50):
#         super().__init__()

#         self.I_zu = nn.Linear(input_dim, hidden_size)
#         self.Z_wg = nn.Linear(hidden_size, hidden_size)
#         self.Z_ug = nn.Linear(input_dim, hidden_size, bias=False)

#         self.G_wz = nn.Linear(hidden_size, hidden_size)
#         self.G_uz = nn.Linear(input_dim, hidden_size, bias=False)

#         self.R_wr = nn.Linear(hidden_size, hidden_size)
#         self.R_ur = nn.Linear(input_dim, hidden_size, bias=False)

#         self.H_wh = nn.Linear(hidden_size, hidden_size)
#         self.H_uh = nn.Linear(input_dim, hidden_size, bias=False)

#         # Non−linear Activation function
#         self.sigma = nn.Tanh()

#     def forward(self, x, s):
#         I = self.I_zu(s)
#         print(f"I shape = {I.shape}")
#         Z = self.sigma(self.Z_wg(I) + self.Z_ug(x))
#         print(f"Z shape = {Z.shape}")
#         G = self.sigma(self.G_wz(Z) + self.G_uz(x))
#         print(f"G shape = {G.shape}")
#         R = self.sigma(self.R_wr(G) + self.R_ur(x))
#         print(f"R shape {R.shape} and s shape {s.shape} and I shape {self.H_wh(I).shape}")
#         H = self.sigma(self.H_wh(I) * R + self.H_uh(x))
#         print(f"H shape {H.shape}")
#         out = (1-G)*H + Z*self.H_wh(I)
#         print(f"out shape {out.shape}")
#         # out = torch.sub(1, G) * H + Z * s
#         return out



In [3]:
class DGMLayer(nn.Module):
    """
    Georgias Detorakis (2024): Practical Aspects on Solving Differential Equations Using Deep Learning: A Primer

    """

    def __init__(self, input_dim=1, hidden_size=50):
        super().__init__()

        self.Z_wg = nn.Linear(hidden_size, hidden_size)
        self.Z_ug = nn.Linear(input_dim, hidden_size, bias=False)

        self.G_wz = nn.Linear(hidden_size, hidden_size)
        self.G_uz = nn.Linear(input_dim, hidden_size, bias=False)

        self.R_wr = nn.Linear(hidden_size, hidden_size)
        self.R_ur = nn.Linear(input_dim, hidden_size, bias=False)

        self.H_wh = nn.Linear(hidden_size, hidden_size)
        self.H_uh = nn.Linear(input_dim, hidden_size, bias=False)

        # Non−linear Activation function
        self.sigma = nn.Tanh()

    def forward(self, x, s):
#         I = self.I_zu(s)
#         print(f"I shape = {I.shape}")
        Z = self.sigma(self.Z_wg(s) + self.Z_ug(x))
        print(f"Z shape = {Z.shape}")
        G = self.sigma(self.G_wz(Z) + self.G_uz(x))
        print(f"G shape = {G.shape}")
        R = self.sigma(self.R_wr(G) + self.R_ur(x))
        print(f"R shape {R.shape} and s shape {s.shape} and I shape {self.H_wh(s).shape}")
        H = self.sigma(self.H_wh(s) * R + self.H_uh(x))
        print(f"H shape {H.shape}")
        out = (1-G)*H + Z*self.H_wh(s)
        print(f"out shape {out.shape}")
        # out = torch.sub(1, G) * H + Z * s
        return out



In [4]:
class DGMLayer0(nn.Module):

    def __init__(self, input_dim=1, hidden_size=50):
        super().__init__()

        self.I_zu = nn.Linear(input_dim, hidden_size)
        self.dgm_layer = DGMLayer(input_dim, hidden_size)
    

    def forward(self, x, s):
        s1 = self.I_zu(s)
        print(f"s1 shape {s1.shape}")
        out = self.dgm_layer(x,s1)
        print(f"out shape {out.shape}")
        return out



In [28]:
class DGMLayerN(nn.Module):

    def __init__(self, input_dim=1, output_dim=1, hidden_size=50):
        super().__init__()

        self.dgm_layer = DGMLayer(input_dim, hidden_size)
        self.K_zu = nn.Linear(hidden_size, output_dim)
    

    def forward(self, x, s):
        
        x1 = self.dgm_layer(x,s)
        print(f"x1 shape {x1.shape}")
        out = self.K_zu(x1)
        print(f"out shape {out.shape}")
        return out

In [29]:
def build_input(q, t_norm, gpos):
    # q: (B,7), t_norm: (B,1), gpos: (B,3)
    return torch.cat([q, t_norm, gpos], dim=-1)

In [30]:
def sample_goals(n):
    # Sample from xyz boundaries (cuboid) from which
    # to train the NN to generate plans to reach points inside this goals cuboid/region
    xs = np.random.uniform(0.25, 0.65, (n, 1))
    ys = np.random.uniform(-0.30, 0.30, (n, 1))
    zs = np.random.uniform(0.10, 0.60, (n, 1))
    return np.hstack([xs, ys, zs]).astype(np.float64)

In [31]:
jmin = np.array([-2.8973, -1.7628, -2.8973, -3.0718, -2.8973, -0.0175, -2.8973], dtype=np.float64)
jmax = np.array([2.8973, 1.7628, 2.8973, -0.0698, 2.8973, 3.7525, 2.8973], dtype=np.float64)
batch = 192
T = 2
Qp = 10

In [32]:
q_np = np.random.uniform(jmin, jmax, (batch, 7)).astype(np.float64)
t_np = np.random.uniform(0.0, T, (batch, 1)).astype(np.float64)
g_np = sample_goals(batch)

In [33]:
g_np.shape

(192, 3)

In [34]:


# running cost via FK (position-only)
# l_np = position_loss_fn(fk, joint_names, batch, Qp, g_np, q_np)

l_np = np.zeros((batch,), dtype=np.float64)
for i in range(batch):
    try:
#         p = fk.ee_position(joint_names, q_np[i])  # fk client gets coordinate position of hand/end-effector
        p = np.random.rand(3) 
        e = p - g_np[i]  # distance between current joint position i and goal position i
        l_np[i] = Qp * float(np.dot(e, e))
    except Exception:
        rospy.logwarn("fk_pos l: couldn't retrieve fk position")
        l_np[i] = 1e3


In [35]:
device = torch.device('cpu')

In [36]:
l_np.shape

(192,)

In [37]:
q = torch.tensor(q_np, dtype=torch.float32, device=device, requires_grad=True)
t = torch.tensor((t_np / T), dtype=torch.float32, device=device, requires_grad=True)
g = torch.tensor(g_np, dtype=torch.float32, device=device)
l = torch.tensor(l_np, dtype=torch.float32, device=device)

In [38]:
print(q.shape)
print(t.shape)
print(g.shape)
print(l.shape)

torch.Size([192, 7])
torch.Size([192, 1])
torch.Size([192, 3])
torch.Size([192])


In [39]:
inp = build_input(q, t, g)

In [40]:
inp.shape

torch.Size([192, 11])

In [41]:
inp.T.shape

torch.Size([11, 192])

In [42]:
dgm_layer = DGMLayer(input_dim=11, hidden_size=192)

In [43]:
dgm_layer

DGMLayer(
  (Z_wg): Linear(in_features=192, out_features=192, bias=True)
  (Z_ug): Linear(in_features=11, out_features=192, bias=False)
  (G_wz): Linear(in_features=192, out_features=192, bias=True)
  (G_uz): Linear(in_features=11, out_features=192, bias=False)
  (R_wr): Linear(in_features=192, out_features=192, bias=True)
  (R_ur): Linear(in_features=11, out_features=192, bias=False)
  (H_wh): Linear(in_features=192, out_features=192, bias=True)
  (H_uh): Linear(in_features=11, out_features=192, bias=False)
  (sigma): Tanh()
)

In [44]:
dgm_layer_0 = DGMLayer0(input_dim=11, hidden_size=192)

In [45]:
dgm_layer_0

DGMLayer0(
  (I_zu): Linear(in_features=11, out_features=192, bias=True)
  (dgm_layer): DGMLayer(
    (Z_wg): Linear(in_features=192, out_features=192, bias=True)
    (Z_ug): Linear(in_features=11, out_features=192, bias=False)
    (G_wz): Linear(in_features=192, out_features=192, bias=True)
    (G_uz): Linear(in_features=11, out_features=192, bias=False)
    (R_wr): Linear(in_features=192, out_features=192, bias=True)
    (R_ur): Linear(in_features=11, out_features=192, bias=False)
    (H_wh): Linear(in_features=192, out_features=192, bias=True)
    (H_uh): Linear(in_features=11, out_features=192, bias=False)
    (sigma): Tanh()
  )
)

In [52]:
init = dgm_layer_0(inp,inp)

s1 shape torch.Size([192, 192])
Z shape = torch.Size([192, 192])
G shape = torch.Size([192, 192])
R shape torch.Size([192, 192]) and s shape torch.Size([192, 192]) and I shape torch.Size([192, 192])
H shape torch.Size([192, 192])
out shape torch.Size([192, 192])
out shape torch.Size([192, 192])


In [47]:
# V = model(build_input(q, t, g))
# loss_pde = hjb_residual_loss(V, q, t, l,
#                              R_inv_diag)  # hjb_residual_loss(V, q, t_norm, running_cost, R_inv_diag)



In [55]:
dgm_layer_n = DGMLayerN(input_dim=11,output_dim=192, hidden_size=192)

In [56]:
dgm_layer_n(inp, init)

Z shape = torch.Size([192, 192])
G shape = torch.Size([192, 192])
R shape torch.Size([192, 192]) and s shape torch.Size([192, 192]) and I shape torch.Size([192, 192])
H shape torch.Size([192, 192])
out shape torch.Size([192, 192])
x1 shape torch.Size([192, 192])
out shape torch.Size([192, 192])


tensor([[-0.8045,  0.2940, -0.1843,  ..., -0.0096,  0.0961,  0.0482],
        [ 0.2087, -0.2738, -0.3764,  ...,  0.0252,  0.6905, -0.3067],
        [ 0.0573, -0.4753,  0.2928,  ..., -0.3591, -1.1449,  0.5138],
        ...,
        [ 0.0375, -0.3404, -0.3520,  ..., -0.1957, -0.5615, -0.4964],
        [-0.4593,  0.2530, -0.3616,  ..., -0.3208,  0.3673, -0.7614],
        [-0.5822,  0.4346,  0.1363,  ...,  0.4099,  0.0049,  0.1128]],
       grad_fn=<AddmmBackward0>)

In [129]:
class ValueNet(nn.Module):
    """
    num_dgm_layers 
    """

    def __init__(self, num_layers=1, input_dim=1, output_dim=1, hidden_size=50):
        super().__init__()
        
        self.layers = nn.ModuleList([DGMLayer0(input_dim, hidden_size)]) + \
        nn.ModuleList([DGMLayer(input_dim,hidden_size) for _ in range(num_layers)]) + \
                      nn.ModuleList([DGMLayerN(input_dim, output_dim, hidden_size)])   
            
#         self.dgm_layer = DGMLayer(input_dim, hidden_size)
    

    def forward(self, x, s):
        
        for i, layer in enumerate(self.layers):
            print(f"layer {i} = {layer}")
            x = layer(s,x)

        print(f"out shape: {x.shape}")
        return x.squeeze(-1)

In [130]:
v_net = ValueNet(num_layers=2, input_dim=11, output_dim=1, hidden_size=192)

In [131]:
v_net

ValueNet(
  (layers): ModuleList(
    (0): DGMLayer0(
      (I_zu): Linear(in_features=11, out_features=192, bias=True)
      (dgm_layer): DGMLayer(
        (Z_wg): Linear(in_features=192, out_features=192, bias=True)
        (Z_ug): Linear(in_features=11, out_features=192, bias=False)
        (G_wz): Linear(in_features=192, out_features=192, bias=True)
        (G_uz): Linear(in_features=11, out_features=192, bias=False)
        (R_wr): Linear(in_features=192, out_features=192, bias=True)
        (R_ur): Linear(in_features=11, out_features=192, bias=False)
        (H_wh): Linear(in_features=192, out_features=192, bias=True)
        (H_uh): Linear(in_features=11, out_features=192, bias=False)
        (sigma): Tanh()
      )
    )
    (1-2): 2 x DGMLayer(
      (Z_wg): Linear(in_features=192, out_features=192, bias=True)
      (Z_ug): Linear(in_features=11, out_features=192, bias=False)
      (G_wz): Linear(in_features=192, out_features=192, bias=True)
      (G_uz): Linear(in_features=11

In [132]:
v = v_net(inp,inp)

layer 0 = DGMLayer0(
  (I_zu): Linear(in_features=11, out_features=192, bias=True)
  (dgm_layer): DGMLayer(
    (Z_wg): Linear(in_features=192, out_features=192, bias=True)
    (Z_ug): Linear(in_features=11, out_features=192, bias=False)
    (G_wz): Linear(in_features=192, out_features=192, bias=True)
    (G_uz): Linear(in_features=11, out_features=192, bias=False)
    (R_wr): Linear(in_features=192, out_features=192, bias=True)
    (R_ur): Linear(in_features=11, out_features=192, bias=False)
    (H_wh): Linear(in_features=192, out_features=192, bias=True)
    (H_uh): Linear(in_features=11, out_features=192, bias=False)
    (sigma): Tanh()
  )
)
s1 shape torch.Size([192, 192])
Z shape = torch.Size([192, 192])
G shape = torch.Size([192, 192])
R shape torch.Size([192, 192]) and s shape torch.Size([192, 192]) and I shape torch.Size([192, 192])
H shape torch.Size([192, 192])
out shape torch.Size([192, 192])
out shape torch.Size([192, 192])
layer 1 = DGMLayer(
  (Z_wg): Linear(in_features=1

In [133]:
v.squeeze(-1).shape

torch.Size([192])

In [134]:
class DGMValueNet(nn.Module):
    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(args, kwargs)

    def init(self, num_layers, input_dim, hidden_size):
        super().__init__()



    pass

In [135]:
import numpy as np

A = np.array([[1, 3], 
              [0, 2]])

B = np.array([[5, 1], 
              [4, 6]])

# Hadamard Product (Element-wise)
hadamard = A * B  

# Standard Matrix Multiplication (Dot Product)
standard = A @ B  


In [136]:
print(hadamard)
print(standard)

[[ 5  3]
 [ 0 12]]
[[17 19]
 [ 8 12]]
